In [ ]:
!pip install torch torchvision torchaudio librosa panns-inference transformers soundfile whisperx -q
!apt-get install -y ffmpeg
!apt-get install -y libcudnn8

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch, librosa, json, os
import numpy as np
from pathlib import Path
from panns_inference import AudioTagging, SoundEventDetection, labels
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
import whisperx
import soundfile as sf

In [ ]:
audio_path = "/content/drive/MyDrive/test.mp3"

In [ ]:
(audio, fs) = librosa.load(audio_path, sr=32000, mono=True)
at = AudioTagging(checkpoint_path=None, device='cuda' if torch.cuda.is_available() else 'cpu')
clipwise_output, embedding = at.inference(audio[np.newaxis, :])

Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth
GPU number: 1


In [ ]:
for i in np.argsort(clipwise_output[0])[::-1][:10]:
    print(f"{labels[i]}: {clipwise_output[0][i]:.3f}")

Speech: 0.841
Music: 0.725
Male speech, man speaking: 0.110
Narration, monologue: 0.092
Television: 0.086
Conversation: 0.037
Chink, clink: 0.032
Speech synthesizer: 0.026
Spray: 0.023
Female speech, woman speaking: 0.018


In [ ]:
sed = SoundEventDetection(checkpoint_path=None, device='cuda' if torch.cuda.is_available() else 'cpu')
framewise_output = sed.inference(audio[np.newaxis, :])

Checkpoint path: /root/panns_data/Cnn14_DecisionLevelMax.pth
GPU number: 1


In [ ]:
segment_len_sec = 1.0
frames_per_second = 100
frames_per_segment = int(segment_len_sec * frames_per_second)
total_frames = framewise_output.shape[1]

In [ ]:
bg_results = []
for start in range(0, total_frames, frames_per_segment):
    end = min(start + frames_per_segment, total_frames)
    segment_output = framewise_output[0, start:end, :]
    mean_scores = segment_output.mean(axis=0)
    detected = [(labels[i], float(mean_scores[i])) for i in range(len(mean_scores)) if mean_scores[i] > 0.3]
    if detected:
        bg_results.append({
            "start": round(start / frames_per_second, 2),
            "end": round(end / frames_per_second, 2),
            "events": detected
        })


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
waveform, sr = librosa.load(audio_path, sr=16000)

model = whisperx.load_model("large-v2", device, compute_type="float16")
audio_whisper = whisperx.load_audio(audio_path)
result = model.transcribe(audio_whisper, batch_size=16)

model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio_whisper, device, return_char_alignments=False)

diarize_model = whisperx.diarize.DiarizationPipeline(
    use_auth_token="YOUR_HUGGING_FACE_TOKEN", device=device)
diarize_segments = diarize_model(audio_whisper)
result = whisperx.assign_word_speakers(diarize_segments, result)

DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _recover
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warn

vocabulary.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/399 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/221 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  std = sequences.std(dim=-1, correction=1)


In [ ]:
emotion_model = AutoModelForAudioClassification.from_pretrained("superb/hubert-large-superb-er").to(device)
emotion_extractor = AutoFeatureExtractor.from_pretrained("superb/hubert-large-superb-er")

for seg in result["segments"]:
    start = seg["start"]
    end = seg["end"]
    clip = waveform[int(start * sr):int(end * sr)]

    if len(clip) < 1000:
        seg["emotion"] = "unknown"
        continue

    inputs = emotion_extractor(clip, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = emotion_model(**inputs).logits
        pred_id = torch.argmax(logits).item()
        seg["emotion"] = emotion_model.config.id2label[pred_id]

segments = result["segments"]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

In [ ]:
def match_bg_event(start, end, bg_events):
    for evt in bg_events:
        if evt["start"] <= start <= evt["end"] or evt["start"] <= end <= evt["end"]:
            top_event = sorted(evt["events"], key=lambda x: x[1], reverse=True)[0]
            return top_event[0]
    return "none"

In [ ]:
scenes = []
for i, seg in enumerate(segments):
    scenes.append({
    "start": round(seg["start"], 3),
    "end": round(seg["end"], 3),
    "text": seg.get("text", ""),
    "speaker": seg.get("speaker", "unknown"),
    "emotion": seg.get("emotion", "neutral"),
    "background_scene": match_bg_event(seg["start"], seg["end"], bg_results),
    "words": seg.get("words", [])
    })

# Сохраняем итог
Path("outputs").mkdir(exist_ok=True)
with open("outputs/scene.json", "w") as f:
    json.dump(scenes, f, indent=2)

print("✅ Успешно! Готовый файл: outputs/scene.json")

✅ Успешно! Готовый файл: outputs/scene.json
